## MAIN JUPYTER NOTE BOOK FOR DISTRIBUTING BATCH PROTEINS

### Utilities and Packages

In [ ]:
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

import matplotlib.pyplot as plt
import warnings
import pickle 
import pandas as pd
import numpy as np

from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split


def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    
    warnings.simplefilter("ignore")
    fxn()

In [ ]:
# Load Olink and Soma assays from the MESA cohort

MESA_Olink  = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", sep = "\t", 
                          compression = "gzip")
MESA_Soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", sep = "\t", 
                        compression = "gzip").drop(columns={"bmi"}, axis=1)

### restricting to shared proteins
common_cols = ["id"] + [col for col in MESA_Soma.columns if col in MESA_Olink.columns and col != "id"]
MESA_Soma  = MESA_Soma[common_cols]
MESA_Olink = MESA_Olink[common_cols]

In [ ]:
# Find subjects who are in both datasets

overlap_IDs = np.unique(list(set(np.array(MESA_Olink["id"])).intersection(set(np.array(MESA_Soma["id"])))))

In [ ]:
# Drop repeated subject IDs in both cohorts

MESA_Olink  = MESA_Olink[MESA_Olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_Soma   = MESA_Soma[MESA_Soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])


print(MESA_Olink.shape)
print(MESA_Soma.shape)

ids1 = list(MESA_Olink["id"])
ids2 = list(MESA_Soma["id"])

# fools check in checking if rows are aligned or not
if ids1 == ids2:
    print("ids in the same order")
else:
    print("ids not in the same order")

# making sure the outcomes index are the same as training index for train test split

In [ ]:
# Compare the Olink and Soma measurements of LEPTIN protin
# # makesure protein list are in the same order for soma and olink
# MESA_Soma_aligned, MESA_Olink_aligned = MESA_Soma.align(MESA_Olink, join='inner', axis=0)

MESA_Soma = MESA_Soma.set_index("id")
MESA_Olink = MESA_Olink.set_index("id")


if MESA_Soma.index.equals(MESA_Olink.index):
    print("yes") 
else:
    print("no")

    
    
print(MESA_Soma.index)
print(MESA_Olink.index)

In [ ]:
X = MESA_Olink # subject id already dropped / moved to index in the last step

protlist = MESA_Soma.columns
protlist = protlist # full protein list for training

print(MESA_Soma.shape)
print(MESA_Olink.shape)
print(len(protlist))

In [ ]:
y_dummy = X.iloc[:, 0]  # or anything for stratification

## get the first column for the row number for indexing, doesn't really matter which column
print(y_dummy)

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y_dummy, random_state=0)

train_index = X_train.index
test_index = X_test.index

train_index1 = y_train.index

# fools proof, double checking
if train_index.equals(train_index1):
    print("yay")
    
# # ### saving splitted train and test index - just need to do this once
# pickle.dump(train_index, open("Data/helper_data/train_index.pkl", "wb"))
# pickle.dump(test_index, open("Data/helper_data/test_index.pkl", "wb"))

## Training loop

In [ ]:
## 2871 proteins for OLINK
## 1737 for overlapping olink
## 4972 proteins for Soma

In [ ]:
# # get all trained proteins so far
# model_dir = "Data/soma_trained_models/"  # adjust as needed

# # List all .pkl files
# trained_files = [f for f in os.listdir(model_dir) if f.endswith(".pkl")]
# trained_proteins = [os.path.splitext(f)[0] for f in trained_files]

# print(f'{trained_proteins} has been trained')
# print(len(trained_proteins))

In [ ]:
## what proteins are left in case the algorithm missed some of the proteins:
# leftover_prots = list(set(protlist) - set(trained_proteins))
# print(leftover_prots[0:10])

## separating into chunks
num_chunks = 10
chunks = [protlist[i::num_chunks] for i in range(num_chunks)]
## saving the prot chunks
for i, chunk in enumerate(chunks):
    k=1+i
    pickle.dump(chunk, open(f"Data/helper_data/protlist_chunk{k}.pkl", "wb"))

In [ ]:
[len(chunk) for chunk in chunks]